In [1]:
import pysam
import re
import numpy as np
import pandas as pd
from collections import defaultdict
from Bio import SeqIO

### Counting the number of insertions, deletions, and substitutions at every position along a reference
The below function counts the number of indels, substitutions, and perfect matches at every position along a reference for the reads aligning to that reference. It outputs the raw counts at each position. 

In [2]:
def count_variants(bam_path):
    bam = pysam.AlignmentFile(bam_path, "rb")
    # 1) pull contig lengths
    header = bam.header.to_dict()
    ref_lengths = {sq["SN"]: sq["LN"] for sq in header["SQ"]}

    # 2) init counters
    stats = {
        ref: {
            "total": 0,
            "cov":   np.zeros(length, dtype=int),
            "del":   np.zeros(length, dtype=int),
            "ins":   np.zeros(length, dtype=int),
            "sub":   np.zeros(length, dtype=int),
        }
        for ref, length in ref_lengths.items()
    }

    # 3–4) scan reads
    for read in bam.fetch(until_eof=True):
        if (not read.is_read1
            or read.is_secondary
            or read.is_supplementary
            or read.mapping_quality < 20
            or read.reference_id != read.next_reference_id
        ):
            continue

        ref = bam.get_reference_name(read.reference_id)
        stats[ref]["total"] += 1

        # 4a) CIGAR
        ref_pos = read.reference_start
        for op, length in read.cigartuples or []:
            if op in (0, 7, 8):
                stats[ref]["cov"][ref_pos:ref_pos+length] += 1
                ref_pos += length
            elif op == 2:
                stats[ref]["del"][ref_pos:ref_pos+length] += 1
                ref_pos += length
            elif op == 1:
                stats[ref]["ins"][ref_pos] += 1
            elif op == 3:
                ref_pos += length

        # 4b) MD tag → substitutions
        if read.has_tag("MD"):
            rpos = read.reference_start
            for num, deleted, sub in re.findall(r'(\d+)|\^([A-Z]+)|([A-Z])',
                                                read.get_tag("MD")):
                if num:
                    rpos += int(num)
                elif deleted:
                    rpos += len(deleted)
                elif sub:
                    stats[ref]["sub"][rpos] += 1
                    rpos += 1

    bam.close()

    # 5) build DataFrame with 'reads' row instead of 'match'
    records = []
    idx     = []
    for ref, data in stats.items():
        length = data["cov"].shape[0]
        reads  = np.full(length, data["total"], dtype=int)
        for var, arr in (
            ("del",   data["del"]),
            ("ins",   data["ins"]),
            ("sub",   data["sub"]),
            ("reads", reads),
        ):
            records.append(arr)
            idx.append((ref, var))

    df = pd.DataFrame(
        records,
        index=pd.MultiIndex.from_tuples(idx, names=("ref", "variant"))
    )
    df.columns = np.arange(1, df.shape[1] + 1)
    return df

In [3]:
# Prepare lists to collect each replicate's DataFrames
r1_list, r2_list, r3_list = [], [], []

with open("../../CRE-deletion-validations_K562-3N-3d-top100/250410_fastq/prefixes.fofn") as prefixes:
    for line in prefixes:
        prefix = line.strip()
        print(prefix)
        sample = prefix.split("_")[0]   # e.g. "EH38D3316369"
        bam_path = f"../../CRE-deletion-validations_K562-3N-3d-top100/250410_alignments/{prefix}.sorted.bam"

        # 1) count all variants
        df = count_variants(bam_path)   # MultiIndex: (ref, variant) × positions

        # 2) restrict to only those contigs whose name starts with your sample ID
        #    (i.e. keep only refs like "EH38D3316369_D…")
        mask = df.index.get_level_values(0).str.startswith(sample)
        df = df.loc[mask]

        # 3) OPTIONAL: if you’d like to keep track of which sample each row came from,
        #    reset the index and add a 'sample' column
        df = (
            df
            .reset_index()                      # columns: ['ref','variant', 1,2,3…]
            .assign(sample=sample)              # add sample column
        )

        # 4) collect by replicate
        if "_R1_" in prefix:
            r1_list.append(df)
        elif "_R2_" in prefix:
            r2_list.append(df)
        elif "_R3_" in prefix:
            r3_list.append(df)

# 5) concatenate—no ignore_index=True so you keep the sample/ref/variant columns
if r1_list:
    r1_df = pd.concat(r1_list, axis=0)
    r1_df.to_csv("./counts/all_variant_counts_r1.MAPQ20.tsv", sep="\t", index=False)

if r2_list:
    r2_df = pd.concat(r2_list, axis=0)
    r2_df.to_csv("./counts/all_variant_counts_r2.MAPQ20.tsv", sep="\t", index=False)

if r3_list:
    r3_df = pd.concat(r3_list, axis=0)
    r3_df.to_csv("./counts/all_variant_counts_r3.MAPQ20.tsv", sep="\t", index=False)

EH38D3316369_R1_S1
EH38D3316369_R2_S2
EH38D3316369_R3_S3
EH38D4003194_R1_S4
EH38D4003194_R2_S5
EH38D4003194_R3_S6
EH38D5328319_R1_S7
EH38D5328319_R2_S8
EH38D5328319_R3_S9
EH38D5356364_R1_S10
EH38D5356364_R2_S11
EH38D5356364_R3_S12
EH38D5793970_R1_S13
EH38D5793970_R2_S14
EH38D5793970_R3_S15
EH38D5895447_R1_S16
EH38D5895447_R2_S17
EH38D5895447_R3_S18
EH38D5977845_R1_S19
EH38D5977845_R2_S20
EH38D5977845_R3_S21
EH38D6094982_R1_S22
EH38D6094982_R2_S23
EH38D6094982_R3_S24


In [2]:
import pysam
from Bio import SeqIO
import pandas as pd

def count_indel_types_full_window(bam_path, fasta_path="./250410_alignments/CRE-deletions.fa"):
    """
    For each edited contig (i.e. both motifs present), count reads with ≥1 indel vs. none
    in three windows:
      – upstream:   [start_pos-50, start_pos]
      – insertion:  [start_pos+1, end_pos-1]
      – downstream: [end_pos, end_pos+50]
    Only tallies a read for a given window if its aligned footprint fully spans that
    window (reference_start <= window_start and reference_end >= window_end) - reads
    that don't reach a window are excluded from that window's denominator entirely,
    rather than defaulting to "perfect".
    Returns a DataFrame with columns:
      ref, location, total_reads, indel_reads, perfect_reads
    """
    # 1) load reference sequences and compute junction coords only if motifs exist
    seqs = {rec.id: str(rec.seq) for rec in SeqIO.parse(fasta_path, "fasta")}
    motif1 = "GTGCCAGTAGCACAGCAATT"
    motif2 = "TAGTTGTGCTTTCCGCG"
    coords = {}
    for ref, seq in seqs.items():
        i1 = seq.find(motif1)
        i2 = seq.find(motif2)
        if i1 < 0 or i2 < 0:
            # skip any WT or incomplete references
            continue
        start_pos = i1 - 1
        end_pos   = i2 + len(motif2) + 8
        coords[ref] = {
            "up":   (max(0, start_pos - 50), start_pos),
            "ins":  (start_pos + 1, end_pos - 1),
            "down": (end_pos, end_pos + 50)
        }
    # 2) init counters only for these edited refs
    regions = ("up", "ins", "down")
    stats = {
        ref: {r: {"total": 0, "indel": 0, "perfect": 0} for r in regions}
        for ref in coords
    }
    # 3) scan BAM, skipping any reads mapping to dropped refs
    bam = pysam.AlignmentFile(bam_path, "rb")
    for read in bam.fetch(until_eof=True):
        if (not read.is_read1
            or read.is_secondary
            or read.is_supplementary
            or read.mapping_quality < 20
            or read.reference_id != read.next_reference_id
        ):
            continue
        ref = bam.get_reference_name(read.reference_id)
        if ref not in stats:
            continue
        # only tally a window if the read's footprint fully spans it
        r_start, r_end = read.reference_start, read.reference_end
        covers = {
            r: (r_start <= s and r_end >= e)
            for r, (s, e) in coords[ref].items()
        }
        for r in regions:
            if covers[r]:
                stats[ref][r]["total"] += 1
        # detect indels overlapping each window
        cig = read.cigartuples or []
        seen = {r: False for r in regions}
        pos = read.reference_start
        for op, length in cig:
            # insertion at pos
            if op == 1:
                for r, (s, e) in coords[ref].items():
                    if s <= pos <= e:
                        seen[r] = True
            # deletion spans [pos, pos+length-1]
            elif op == 2:
                for r, (s, e) in coords[ref].items():
                    if not seen[r] and (pos <= e and pos + length - 1 >= s):
                        seen[r] = True
                pos += length
                continue
            # advance reference position if op consumes ref
            if op in (0, 2, 3, 7, 8):
                pos += length
        # tally - only for windows this read actually covers
        for r in regions:
            if not covers[r]:
                continue
            if seen[r]:
                stats[ref][r]["indel"]   += 1
            else:
                stats[ref][r]["perfect"] += 1
    bam.close()
    # 4) assemble DataFrame
    rows = []
    for ref, reg in stats.items():
        for r, ctr in reg.items():
            rows.append({
                "ref": ref,
                "location": {"up":"upstream","ins":"insertion","down":"downstream"}[r],
                "total_reads": ctr["total"],
                "indel_reads": ctr["indel"],
                "perfect_reads": ctr["perfect"]
            })
    return pd.DataFrame(rows, columns=[
        "ref","location","total_reads","indel_reads","perfect_reads"
    ])

In [3]:
# Prepare lists to collect each replicate's DataFrames
r1_list, r2_list, r3_list = [], [], []

with open("./250410_fastq/prefixes.fofn") as prefixes:
    for line in prefixes:
        prefix = line.strip()
        sample = prefix.split("_")[0]   # e.g. "EH38D3316369"
        bam_path = f"./250410_alignments/{prefix}.sorted.bam"

        print(prefix)
        # 1) count all variants
        df = count_indel_types_full_window(bam_path)  

        # 2) restrict to only those contigs whose name starts with your sample ID
        #    (i.e. keep only refs like "EH38D3316369_D…")
        df = df[df['ref'].str.contains(sample)]

        # 4) collect by replicate
        if "_R1_" in prefix:
            r1_list.append(df)
        elif "_R2_" in prefix:
            r2_list.append(df)
        elif "_R3_" in prefix:
            r3_list.append(df)

# 5) concatenate—no ignore_index=True so you keep the sample/ref/variant columns
if r1_list:
    r1_df = pd.concat(r1_list, axis=0)
    r1_df.to_csv("./counts/indel_types_r1.MAPQ20.fullwindow.tsv", sep="\t", index=True)

if r2_list:
    r2_df = pd.concat(r2_list, axis=0)
    r2_df.to_csv("./counts/indel_types_r2.MAPQ20.fullwindow.tsv", sep="\t", index=True)

if r3_list:
    r3_df = pd.concat(r3_list, axis=0)
    r3_df.to_csv("./counts/indel_types_r3.MAPQ20.fullwindow.tsv", sep="\t", index=True)

EH38D3316369_R1_S1
EH38D3316369_R2_S2
EH38D3316369_R3_S3
EH38D4003194_R1_S4
EH38D4003194_R2_S5
EH38D4003194_R3_S6
EH38D5328319_R1_S7
EH38D5328319_R2_S8
EH38D5328319_R3_S9
EH38D5356364_R1_S10
EH38D5356364_R2_S11
EH38D5356364_R3_S12
EH38D5793970_R1_S13
EH38D5793970_R2_S14
EH38D5793970_R3_S15
EH38D5895447_R1_S16
EH38D5895447_R2_S17
EH38D5895447_R3_S18
EH38D5977845_R1_S19
EH38D5977845_R2_S20
EH38D5977845_R3_S21
EH38D6094982_R1_S22
EH38D6094982_R2_S23
EH38D6094982_R3_S24
